# UMAG — Amazon Books v3 (DIEN-Aligned Preprocessing)


## Cell 1 — Imports & Config

In [ ]:
import ast, json, math, time, hashlib, random, os
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import IterableDataset, DataLoader
from tqdm import tqdm
from sklearn.metrics import roc_auc_score
from pathlib import Path

# Add project root to path so src/ is importable
sys.path.insert(0, str(Path('..').resolve()))
from src.utils.paths import P

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_ROOT   = Path("/Volumes/T5 EVO/hf/amazon_books")
REVIEWS     = DATA_ROOT / "reviews_Books_5.json"
assert REVIEWS.exists(), f"Not found: {REVIEWS}"

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)


RATING_THRESHOLD = 4          # discard reviews below this rating
RATING_WEIGHT    = {5: 1.00, 4: 0.60}   # weights for positives only

# ── Tier thresholds (Books-recalibrated, same as v2) ─────────────────────────
TIER_FN    = lambda n: 0 if n <= 8 else (1 if n <= 20 else (2 if n <= 50 else 3))
TIER_NAMES = ["cold", "casual", "active", "power"]

CFG = {
    # ── Embedding buckets ─────────────────────────────────────────────────────
    "user_buckets":      2_000_000,
    "item_buckets":      1_500_000,    # Increased from v2 (Books ~800K items)

    # ── Model — IDENTICAL to Electronics v6 / Books v2 ────────────────────────
    "embed_dim":         48,
    "hidden":            128,
    "dropout":           0.2,
    "n_tier_emb":        8,

    # ── Temporal windows ──────────────────────────────────────────────────────
    # session=last 3, short=next 7 (items 4–10), long=all older
    # Total context depth: covers DIEN's 20-item window
    "session_window":    3,
    "short_window":      7,
    "long_window":       50,

    # ── Label — binary BPR ────────────────────────────────────────────────────
    "label_mode":        "binary",
    "rating_weight":     RATING_WEIGHT,
    "rating_threshold":  RATING_THRESHOLD,
    "neg_per_pos":       1,

    # ── Training ──────────────────────────────────────────────────────────────
    "lr":                3e-4,
    "weight_decay":      1e-5,
    "epochs":            4,
    "batch_size":        2048,
    "num_workers":       0,

    # ── Sliding-window cap (None = all prefixes) ──────────────────────────────
    # Set to an int to limit samples per user (useful for smoke tests).
    # None means all N-1 prefixes per user (recommended for full training).
    "max_samples_per_user": None,

    "label_smoothing":   0.05,
    "use_temporal_feat": True,

    # ── Checkpointing ─────────────────────────────────────────────────────────
    "ckpt_dir":          str(Path.cwd() / "checkpoints_umag_books_v3"),
    "ckpt_every_steps":  1000,
    "resume":            False,
    "random_state":      42,
}

Path(CFG["ckpt_dir"]).mkdir(parents=True, exist_ok=True)
torch.manual_seed(CFG["random_state"])
np.random.seed(CFG["random_state"])
random.seed(CFG["random_state"])

print("Config ready — Books v3 (DIEN-aligned preprocessing)")
print(f"  Rating threshold : ≥{RATING_THRESHOLD}★ only (1–3★ discarded)")
print(f"  Split strategy   : leave-one-out, all users train")
print(f"  Augmentation     : sliding window (all N-1 prefixes per user)")
print(f"  item_buckets     : {CFG['item_buckets']:,}")
print(f"  embed_dim        : {CFG['embed_dim']}")
print(f"  DIEN benchmark   : AUC=0.8453")

Device: cpu
Config ready — Books v3 (DIEN-aligned preprocessing)
  Rating threshold : ≥4★ only (1–3★ discarded)
  Split strategy   : leave-one-out, all users train
  Augmentation     : sliding window (all N-1 prefixes per user)
  item_buckets     : 1,500,000
  embed_dim        : 48
  DIEN benchmark   : AUC=0.8453


## Cell 2 — Full History Builder


In [2]:
def stable_hash(x, buckets):
    if x is None: x = ""
    if not isinstance(x, str): x = str(x)
    h = hashlib.blake2b(x.encode("utf-8"), digest_size=8).digest()
    return int.from_bytes(h, "little") % buckets

def parse_line(line):
    line = line.strip()
    if not line: return None
    try:    return json.loads(line)
    except:
        try: return ast.literal_eval(line)
        except: return None

SECS_PER_DAY = 86400.0
PAD = 0

print("Building user history (4★+ only — DIEN-aligned) ...")
print("  Run caffeinate -dims before this cell.")
t0 = time.time()

raw_history     = defaultdict(list)
star_counts     = Counter()
discarded_low   = 0
all_item_hashes = set()
total = skipped = 0
rw_map    = CFG["rating_weight"]
ib        = CFG["item_buckets"]
threshold = CFG["rating_threshold"]

with open(REVIEWS, "r", encoding="utf-8") as f:
    for line in f:
        rec = parse_line(line)
        if rec is None:
            skipped += 1
            continue
        uid     = rec.get("reviewerID")
        asin    = rec.get("asin")
        ts      = rec.get("unixReviewTime")
        overall = rec.get("overall")
        if None in (uid, asin, ts, overall):
            skipped += 1
            continue
        total += 1
        stars  = int(round(float(overall)))
        star_counts[stars] += 1

        # KEY CHANGE v3: discard low-rating reviews
        if stars < threshold:
            discarded_low += 1
            continue

        rw     = float(rw_map.get(stars, 0.60))
        i_hash = stable_hash(str(asin), ib)
        all_item_hashes.add(i_hash)
        raw_history[uid].append((int(ts), i_hash, rw))

# Sort each user's history chronologically
for uid in raw_history:
    raw_history[uid].sort(key=lambda x: x[0])

# Keep only users with ≥2 positive interactions (need at least 1 context + 1 target)
user_history = {uid: h for uid, h in raw_history.items() if len(h) >= 2}
user_tier    = {uid: TIER_FN(len(h)) for uid, h in user_history.items()}
ALL_ITEMS    = sorted(all_item_hashes)

print(f"Done in {time.time()-t0:.1f}s")
print(f"Total lines    : {total:,}  |  Skipped (parse): {skipped:,}")
print(f"Discarded <{threshold}★   : {discarded_low:,}  ({discarded_low/total*100:.1f}%)")
print(f"Users kept     : {len(user_history):,}  (≥2 positive reviews)")
print(f"Unique items   : {len(ALL_ITEMS):,}")

print("\nRating distribution (full corpus):")
for s in sorted(star_counts):
    marker = " ✓ KEPT" if s >= threshold else " ✗ discarded"
    print(f"  {s}★: {star_counts[s]:>9,}  ({star_counts[s]/total*100:.1f}%){marker}")

tc = Counter(user_tier.values())
print("\nUser tiers (4★+ filtered corpus):")
for i, name in enumerate(TIER_NAMES):
    c = tc[i]
    avg_len = np.mean([len(user_history[u]) for u, t in user_tier.items() if t == i]) if c > 0 else 0
    print(f"  {name:<8}: {c:>8,} ({c/len(user_tier)*100:.1f}%)  avg_reviews={avg_len:.1f}")

# Estimate augmented training samples
total_prefixes = sum(len(h) - 1 for h in user_history.values())
print(f"\nSliding window estimate:")
print(f"  Total positive samples (all prefixes): {total_prefixes:,}")
print(f"  With neg_per_pos=1: {total_prefixes*2:,} samples total")
print(f"  At batch_size=2048: ~{math.ceil(total_prefixes*2/CFG['batch_size']):,} iterations/epoch")

Building user history (4★+ only — DIEN-aligned) ...
  Run caffeinate -dims before this cell.
Done in 141.1s
Total lines    : 8,898,041  |  Skipped (parse): 0
Discarded <4★   : 1,694,132  (19.0%)
Users kept     : 593,380  (≥2 positive reviews)
Unique items   : 325,953

Rating distribution (full corpus):
  1★:   323,833  (3.6%) ✗ discarded
  2★:   415,110  (4.7%) ✗ discarded
  3★:   955,189  (10.7%) ✗ discarded
  4★: 2,223,094  (25.0%) ✓ KEPT
  5★: 4,980,815  (56.0%) ✓ KEPT

User tiers (4★+ filtered corpus):
  cold    :  398,170 (67.1%)  avg_reviews=5.3
  casual  :  138,324 (23.3%)  avg_reviews=12.5
  active  :   39,684 (6.7%)  avg_reviews=30.4
  power   :   17,202 (2.9%)  avg_reviews=126.5

Sliding window estimate:
  Total positive samples (all prefixes): 6,602,814
  With neg_per_pos=1: 13,205,628 samples total
  At batch_size=2048: ~6,449 iterations/epoch


## Cell 3 — Train / Val Split


In [ ]:
train_history = {}
val_history_full = {}

for uid, hist in user_history.items():
    train_history[uid]    = hist[:-1]   # context for training (may be empty for len=2)
    val_history_full[uid] = hist        # full history; last item = target


VAL_EVAL_CAP = 5_000
rng_cap = random.Random(CFG["random_state"] + 1)
val_sample_keys = []
for tier_idx in range(4):
    tier_keys = [u for u in val_history_full if user_tier.get(u, 1) == tier_idx]
    rng_cap.shuffle(tier_keys)
    n = max(1, int(len(tier_keys) / len(val_history_full) * VAL_EVAL_CAP))
    val_sample_keys.extend(tier_keys[:n])
val_history = {u: val_history_full[u] for u in val_sample_keys}


n_train_samples = sum(len(h) for h in train_history.values())  # = total prefixes
est_iters = math.ceil(n_train_samples * (1 + CFG["neg_per_pos"]) / CFG["batch_size"])

print(f"Train users         : {len(train_history):,}  (ALL users)")
print(f"Val users (full)    : {len(val_history_full):,}")
print(f"Val users (capped)  : {len(val_history):,}  (per-epoch eval)")
print(f"Train pos samples   : {n_train_samples:,}  (sliding window)")
print(f"Total samples/epoch : {n_train_samples*(1+CFG['neg_per_pos']):,}  (pos + neg)")
print(f"Iterations/epoch    : ~{est_iters:,}")
print(f"Total iterations    : ~{est_iters*CFG['epochs']:,}  ({CFG['epochs']} epochs)")

# Sanity check: verify no data leakage
assert all(len(h) >= 1 for h in train_history.values()), \
    "Some train users have empty history — check filter"
print("\nSanity checks passed.")

Train users         : 593,380  (ALL users)
Val users (full)    : 593,380
Val users (capped)  : 4,998  (per-epoch eval)
Train pos samples   : 6,602,814  (sliding window)
Total samples/epoch : 13,205,628  (pos + neg)
Iterations/epoch    : ~6,449
Total iterations    : ~25,796  (4 epochs)

Sanity checks passed.


## Cell 4 — Sliding-Window Streaming Dataset


In [ ]:
def pad_or_trim(seq, length):
    if len(seq) >= length: return seq[-length:]
    return [PAD] * (length - len(seq)) + seq

def pad_or_trim_w(seq, length):
    if len(seq) >= length: return seq[-length:]
    return [0.0] * (length - len(seq)) + seq


class SlidingWindowUMAGDataset(IterableDataset):
    """
    DIEN-aligned sliding-window dataset.

    For each user u with training history [i1, i2, ..., iK]:
      Generates K positive samples:
        pos=1: context=[]         → target=i1
        pos=2: context=[i1]       → target=i2
        ...
        pos=K: context=[i1..iK-1] → target=iK

    Each positive gets neg_per_pos random negatives (uniform, excluding seen items).
    User order is shuffled each epoch. Architecture inputs are identical to v2.
    """

    def __init__(self, cfg, train_history, user_tier, all_items, epoch=0):
        super().__init__()
        self.cfg           = cfg
        self.train_history = train_history
        self.user_tier     = user_tier
        self.all_items     = all_items
        self.epoch         = epoch
        self.S   = cfg["session_window"]
        self.M   = cfg["short_window"]
        self.L   = cfg["long_window"]
        self.ssl = self.M - self.S   # short slot length (between session and long)

    def __iter__(self):
        cfg   = self.cfg
        ub    = cfg["user_buckets"]
        bs    = cfg["batch_size"]
        S, M, L, ssl = self.S, self.M, self.L, self.ssl
        neg_k = cfg["neg_per_pos"]
        max_spu = cfg["max_samples_per_user"]   # None = all
        all_items_arr = self.all_items
        n_items       = len(all_items_arr)
        rng = random.Random(cfg["random_state"] + self.epoch * 9999)

        # Shuffle user order each epoch
        uid_list = list(self.train_history.keys())
        rng.shuffle(uid_list)

        buffers = {
            "user": [], "item": [], "tier": [],
            "sess": [], "sess_w": [],
            "short": [], "short_w": [],
            "long": [], "long_w": [],
            "tfeat": [], "y": []
        }

        def flush():
            b = buffers
            batch = (
                torch.tensor(b["user"],    dtype=torch.long),
                torch.tensor(b["item"],    dtype=torch.long),
                torch.tensor(b["tier"],    dtype=torch.long),
                torch.tensor(b["sess"],    dtype=torch.long),
                torch.tensor(b["sess_w"],  dtype=torch.float32),
                torch.tensor(b["short"],   dtype=torch.long),
                torch.tensor(b["short_w"], dtype=torch.float32),
                torch.tensor(b["long"],    dtype=torch.long),
                torch.tensor(b["long_w"],  dtype=torch.float32),
                torch.tensor(b["tfeat"],   dtype=torch.float32),
                torch.tensor(b["y"],       dtype=torch.float32),
            )
            for k in buffers: buffers[k] = []
            return batch

        def add_sample(u_hash, it_hash, t_idx, past_items, past_ws, time_feat, label):
            # Split past_items into three temporal windows (identical to v2)
            sess_items  = past_items[-S:]
            sess_ws_    = past_ws[-S:]
            short_items = past_items[-(M):-S] if len(past_items) > S else []
            short_ws_   = past_ws[-(M):-S]    if len(past_ws)    > S else []
            long_items  = past_items[:-(M)]    if len(past_items) > M else []
            long_ws_    = past_ws[:-(M)]       if len(past_ws)    > M else []

            sess_pad  = pad_or_trim(sess_items,   S)
            sess_w_p  = pad_or_trim_w(sess_ws_,   S)
            short_pad = pad_or_trim(short_items, ssl)
            short_w_p = pad_or_trim_w(short_ws_, ssl)
            long_pad  = pad_or_trim(long_items,   L)
            long_w_p  = pad_or_trim_w(long_ws_,   L)

            # Fallback: if session is all-PAD, borrow from short
            if all(x == PAD for x in sess_pad):
                sess_pad = short_pad[:S] if len(short_pad) >= S \
                           else short_pad + [PAD]*(S-len(short_pad))
                sess_w_p = short_w_p[:S] if len(short_w_p) >= S \
                           else short_w_p + [0.0]*(S-len(short_w_p))

            buffers["user"].append(u_hash)
            buffers["item"].append(it_hash)
            buffers["tier"].append(t_idx)
            buffers["sess"].append(sess_pad)
            buffers["sess_w"].append(sess_w_p)
            buffers["short"].append(short_pad)
            buffers["short_w"].append(short_w_p)
            buffers["long"].append(long_pad)
            buffers["long_w"].append(long_w_p)
            buffers["tfeat"].append(time_feat)
            buffers["y"].append(label)

        
        for uid in uid_list:
            hist   = self.train_history[uid]   # (ts, i_hash, rw) tuples, sorted
            u_hash = stable_hash(uid, ub)
            t_idx  = self.user_tier.get(uid, 1)

            
            seen_set = set(h[1] for h in hist)

            samples_emitted = 0


            for pos in range(len(hist)):
                if max_spu is not None and samples_emitted >= max_spu:
                    break

                target_entry = hist[pos]
                it_hash      = target_entry[1]
                ts           = target_entry[0]

                # Context = all items BEFORE this position
                context    = hist[:pos]
                past_items = [h[1] for h in context]
                past_ws    = [h[2] for h in context]

                # Temporal feature: log days since last interaction
                if context:
                    last_ts    = context[-1][0]
                    days_delta = max(0.0, (ts - last_ts) / SECS_PER_DAY)
                    time_feat  = math.log1p(days_delta)
                else:
                    time_feat = 0.0

                # Positive sample
                add_sample(u_hash, it_hash, t_idx, past_items, past_ws, time_feat, 1.0)
                samples_emitted += 1

                # Negative samples (uniform random, exclude seen)
                negs_emitted = 0
                attempts     = 0
                while negs_emitted < neg_k and attempts < neg_k * 30:
                    attempts += 1
                    neg_item = all_items_arr[rng.randrange(n_items)]
                    if neg_item not in seen_set:
                        add_sample(u_hash, neg_item, t_idx, past_items, past_ws,
                                   time_feat, 0.0)
                        negs_emitted += 1

                if len(buffers["y"]) >= bs:
                    yield flush()

        # Emit remaining partial batch
        if buffers["y"]:
            yield flush()


# ── Smoke test ────────────────────────────────────────────────────────────────
print("Smoke test (2 batches, 10 users max) ...")
smoke_cfg = {**CFG, "max_samples_per_user": 5}   # 5 samples per user
# Use first 100 users from train_history for speed
smoke_hist = dict(list(train_history.items())[:100])
smoke_ds   = SlidingWindowUMAGDataset(smoke_cfg, smoke_hist, user_tier, ALL_ITEMS, epoch=0)
n_smoke = 0
for batch in DataLoader(smoke_ds, batch_size=None, num_workers=0):
    u, i, t, s, sw, sh, shw, l, lw, tf, y = batch
    if n_smoke == 0:
        print(f"  Batch shapes — user:{u.shape} sess:{s.shape} short:{sh.shape} long:{l.shape}")
        print(f"  time_feat: mean={tf.mean():.3f}  max={tf.max():.3f}")
        pos_rate = float((y==1).float().mean())
        print(f"  Label balance: pos_rate={pos_rate:.3f}  (expected ~0.5)")
        assert abs(pos_rate - 0.5) < 0.15, f"Label balance off: {pos_rate:.3f}"
        # Index bounds check
        assert i.max().item() < CFG["item_buckets"], \
            f"item index {i.max().item()} >= item_buckets {CFG['item_buckets']}"
        assert u.max().item() < CFG["user_buckets"], \
            f"user index {u.max().item()} >= user_buckets {CFG['user_buckets']}"
        assert s.max().item() <= CFG["item_buckets"], \
            f"sess index {s.max().item()} > item_buckets"
    n_smoke += 1
    if n_smoke >= 2: break
print(f"Smoke test passed ({n_smoke} batches).")

Smoke test (2 batches, 10 users max) ...
  Batch shapes — user:torch.Size([934]) sess:torch.Size([934, 3]) short:torch.Size([934, 4]) long:torch.Size([934, 50])
  time_feat: mean=2.192  max=8.475
  Label balance: pos_rate=0.500  (expected ~0.5)
Smoke test passed (1 batches).



## Cell 5 — Model 


In [5]:
class WeightedScaleAttention(nn.Module):
    """Identical to MerRec UMAGv2 / Electronics v6 / Books v2."""
    def __init__(self, embed_dim):
        super().__init__()
        self.scale = math.sqrt(embed_dim)
        self.W_q   = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k   = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v   = nn.Linear(embed_dim, embed_dim, bias=False)
        self.ln    = nn.LayerNorm(embed_dim)

    def forward(self, query, keys, event_weights, key_padding_mask=None):
        q      = self.W_q(query).unsqueeze(1)
        k      = self.W_k(keys)
        v      = self.W_v(keys)
        scores = torch.bmm(q, k.transpose(1, 2)) / self.scale
        ew_prior = torch.log(event_weights.clamp(min=1e-6)).unsqueeze(1)
        scores   = scores + ew_prior
        if key_padding_mask is not None:
            scores = scores.masked_fill(key_padding_mask.unsqueeze(1), float("-inf"))
        attn_w  = torch.nan_to_num(torch.softmax(scores, dim=-1), nan=0.0)
        context = self.ln(torch.bmm(attn_w, v).squeeze(1))
        return context, attn_w.squeeze(1)


class UMAGv2(nn.Module):
    """
    Exact MerRec UMAGv2 — identical to Electronics v6 and Books v2.
    Gate input: pure 3-context concatenation (no tier — preserves interpretability).
    Tier enters only the MLP tower.
    """
    def __init__(self, user_buckets, item_buckets, embed_dim, hidden,
                 n_tier_emb=8, dropout=0.2, use_temporal_feat=True):
        super().__init__()
        D = embed_dim
        self.use_temporal_feat = use_temporal_feat

        self.item_emb = nn.Embedding(item_buckets + 1, D, padding_idx=0)
        self.user_emb = nn.Embedding(user_buckets, D)
        self.tier_emb = nn.Embedding(4, n_tier_emb)

        self.attn_session = WeightedScaleAttention(D)
        self.attn_short   = WeightedScaleAttention(D)
        self.attn_long    = WeightedScaleAttention(D)

        # Scale gate: input is pure 3D context concat (tier excluded → interpretable)
        self.scale_gate = nn.Sequential(
            nn.Linear(D * 3 + n_tier_emb, 64),
            nn.ReLU(),
            nn.Linear(64, 3),
            nn.Softmax(dim=-1),
        )

        temporal_dim = 1 if use_temporal_feat else 0
        mlp_in = D * 3 + n_tier_emb + temporal_dim
        self.ln_mlp = nn.LayerNorm(mlp_in)
        self.mlp = nn.Sequential(
            nn.Linear(mlp_in, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )

        # Wide (linear) component
        self.user_lin = nn.Embedding(user_buckets, 1)
        self.item_lin = nn.Embedding(item_buckets + 1, 1)
        self.bias     = nn.Parameter(torch.zeros(1))

        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.tier_emb.weight, std=0.01)
        nn.init.zeros_(self.user_lin.weight)
        nn.init.zeros_(self.item_lin.weight)
        with torch.no_grad(): self.item_emb.weight[0].fill_(0)

    def forward(self, user_idx, item_idx, tier_idx,
                sess_seq, sess_w, short_seq, short_w, long_seq, long_w,
                time_feat=None, return_weights=False):

        u_emb  = self.user_emb(user_idx)
        it_emb = self.item_emb(item_idx)
        t_emb  = self.tier_emb(tier_idx)

        sess_mask  = (sess_seq  == 0)
        short_mask = (short_seq == 0)
        long_mask  = (long_seq  == 0)
        sess_w  = sess_w  * (~sess_mask).float()
        short_w = short_w * (~short_mask).float()
        long_w  = long_w  * (~long_mask).float()

        ctx_sess,  w_sess  = self.attn_session(
            it_emb, self.item_emb(sess_seq),  sess_w,  sess_mask)
        ctx_short, w_short = self.attn_short(
            it_emb, self.item_emb(short_seq), short_w, short_mask)
        ctx_long,  w_long  = self.attn_long(
            it_emb, self.item_emb(long_seq),  long_w,  long_mask)

        gate_input = torch.cat([ctx_sess, ctx_short, ctx_long, t_emb], dim=-1)
        gates      = self.scale_gate(gate_input)

        gated_ctx = (
            gates[:,0:1] * ctx_sess  +
            gates[:,1:2] * ctx_short +
            gates[:,2:3] * ctx_long
        )

        mlp_parts = [u_emb, it_emb, gated_ctx, t_emb]
        if self.use_temporal_feat and time_feat is not None:
            mlp_parts.append(time_feat.unsqueeze(-1))
        mlp_input = self.ln_mlp(torch.cat(mlp_parts, dim=-1))
        deep = self.mlp(mlp_input).squeeze(-1)
        wide = (
            self.user_lin(user_idx).squeeze(-1) +
            self.item_lin(item_idx).squeeze(-1) +
            self.bias
        )

        pred = torch.sigmoid(deep + wide)

        if return_weights:
            return pred, gates, w_sess, w_short, w_long
        return pred


model = UMAGv2(
    user_buckets=CFG["user_buckets"],
    item_buckets=CFG["item_buckets"],
    embed_dim=CFG["embed_dim"],
    hidden=CFG["hidden"],
    n_tier_emb=CFG["n_tier_emb"],
    dropout=CFG["dropout"],
    use_temporal_feat=CFG["use_temporal_feat"],
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"UMAGv2 parameters: {n_params:,}")
print("Architecture: IDENTICAL to MerRec UMAGv2 / Electronics v6 / Books v2")

# Embedding bounds safety check
assert (CFG["item_buckets"] - 1) < model.item_emb.num_embeddings, \
    f"item_buckets={CFG['item_buckets']} exceeds embedding size {model.item_emb.num_embeddings}"
assert CFG["user_buckets"] <= model.user_emb.num_embeddings, \
    f"user_buckets={CFG['user_buckets']} exceeds user embedding size"
print(f"Bounds check: item_emb={model.item_emb.num_embeddings:,}  "
      f"user_emb={model.user_emb.num_embeddings:,}  ✓")

UMAGv2 parameters: 171,559,432
Architecture: IDENTICAL to MerRec UMAGv2 / Electronics v6 / Books v2
Bounds check: item_emb=1,500,001  user_emb=2,000,000  ✓



## Cell 6 — Loss Function



In [6]:
class SmoothedBCELoss(nn.Module):
    def __init__(self, smoothing=0.05):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, pred, target):
        target_s = target * (1 - self.smoothing) + 0.5 * self.smoothing
        return F.binary_cross_entropy(pred, target_s)


loss_fn = SmoothedBCELoss(smoothing=CFG["label_smoothing"])
print("Loss: SmoothedBCE(0.05) — identical to Electronics v6 / Books v2")

Loss: SmoothedBCE(0.05) — identical to Electronics v6 / Books v2



## Cell 7 — Validation: Leave-One-Out + Ranking Metrics



In [10]:
def compute_ndcg_hr(ranks, k=10):
    hits  = [1 if r < k else 0 for r in ranks]
    ndcgs = [1.0 / math.log2(r + 2) if r < k else 0.0 for r in ranks]
    return float(np.mean(ndcgs)), float(np.mean(hits))


@torch.no_grad()
def run_val(model, val_history, user_tier, all_items, cfg, device):
    """DIEN-aligned LOO evaluation. Returns (auc, ndcg@10, hr@10, n_users)."""
    model.eval()
    # Cap at 5000 for speed
    val_history = dict(list(val_history.items())[:5_000])
    rng_val       = random.Random(42)
    all_items_arr = all_items
    n_items       = len(all_items_arr)
    N_NEG_LOO     = 99

    ub  = cfg["user_buckets"]
    S   = cfg["session_window"]
    M   = cfg["short_window"]
    L   = cfg["long_window"]
    ssl = M - S

    auc_preds, auc_labels = [], []
    loo_ranks = []

    for uid, hist in val_history.items():
        if len(hist) < 2:
            continue

        u_hash    = stable_hash(uid, ub)
        t_idx     = user_tier.get(uid, 1)
        context   = hist[:-1]          # all but last
        pos_entry = hist[-1]           # last item = test target
        pos_item  = pos_entry[1]

        past_items = [h[1] for h in context]
        past_ws    = [h[2] for h in context]

        last_ts    = context[-1][0] if context else pos_entry[0]
        days_delta = max(0.0, (pos_entry[0] - last_ts) / SECS_PER_DAY)
        time_feat  = math.log1p(days_delta)

        # Sample 99 negatives (exclude all items user interacted with)
        seen_set  = set(h[1] for h in hist)
        neg_items = []
        attempts  = 0
        while len(neg_items) < N_NEG_LOO and attempts < N_NEG_LOO * 20:
            attempts += 1
            cand = all_items_arr[rng_val.randrange(n_items)]
            if cand not in seen_set:
                neg_items.append(cand)
        if len(neg_items) < N_NEG_LOO:
            continue

        candidates = [pos_item] + neg_items
        n_cand     = len(candidates)

        # Build context tensors
        sess_items  = past_items[-S:]
        sess_ws_    = past_ws[-S:]
        short_items = past_items[-(M):-S] if len(past_items) > S else []
        short_ws_   = past_ws[-(M):-S]    if len(past_ws)    > S else []
        long_items  = past_items[:-(M)]    if len(past_items) > M else []
        long_ws_    = past_ws[:-(M)]       if len(past_ws)    > M else []

        sess_pad  = pad_or_trim(sess_items,   S)
        short_pad = pad_or_trim(short_items, ssl)
        long_pad  = pad_or_trim(long_items,   L)
        sess_w_p  = pad_or_trim_w(sess_ws_,   S)
        short_w_p = pad_or_trim_w(short_ws_, ssl)
        long_w_p  = pad_or_trim_w(long_ws_,   L)

        if all(x == PAD for x in sess_pad):
            sess_pad = short_pad[:S] if len(short_pad) >= S \
                       else short_pad + [PAD]*(S-len(short_pad))
            sess_w_p = short_w_p[:S] if len(short_w_p) >= S \
                       else short_w_p + [0.0]*(S-len(short_w_p))

        u_t   = torch.tensor([u_hash]   * n_cand, dtype=torch.long).to(device)
        i_t   = torch.tensor(candidates,           dtype=torch.long).to(device)
        ti_t  = torch.tensor([t_idx]    * n_cand, dtype=torch.long).to(device)
        s_t   = torch.tensor([sess_pad]  * n_cand, dtype=torch.long).to(device)
        sw_t  = torch.tensor([sess_w_p]  * n_cand, dtype=torch.float32).to(device)
        sh_t  = torch.tensor([short_pad] * n_cand, dtype=torch.long).to(device)
        shw_t = torch.tensor([short_w_p] * n_cand, dtype=torch.float32).to(device)
        l_t   = torch.tensor([long_pad]  * n_cand, dtype=torch.long).to(device)
        lw_t  = torch.tensor([long_w_p]  * n_cand, dtype=torch.float32).to(device)
        tf_t  = torch.tensor([time_feat] * n_cand, dtype=torch.float32).to(device)

        scores    = model(u_t, i_t, ti_t, s_t, sw_t, sh_t, shw_t, l_t, lw_t, tf_t)
        scores_np = scores.cpu().numpy()

        rank = int((scores_np[1:] >= scores_np[0]).sum())
        loo_ranks.append(rank)
        auc_preds.extend(scores_np.tolist())
        auc_labels.extend([1.0] + [0.0] * N_NEG_LOO)

    auc        = roc_auc_score(auc_labels, auc_preds) if auc_labels else float("nan")
    ndcg10, hr10 = compute_ndcg_hr(loo_ranks, k=10)

    model.train()
    return auc, ndcg10, hr10, len(loo_ranks)


print("Validation function ready.")
print("  Protocol: LOO, 99 negatives, AUC + NDCG@10 + HR@10")
print("  Identical to DIEN evaluation protocol")

Validation function ready.
  Protocol: LOO, 99 negatives, AUC + NDCG@10 + HR@10
  Identical to DIEN evaluation protocol


## Cell 8 — Training Loop



In [ ]:
opt = torch.optim.AdamW(model.parameters(),
                        lr=CFG["lr"], weight_decay=CFG["weight_decay"])

# Cosine LR — T_max based on sliding-window sample count
EXPECTED_STEPS = int(
    n_train_samples * (1 + CFG["neg_per_pos"]) /
    CFG["batch_size"] * CFG["epochs"]
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt, T_max=max(1, EXPECTED_STEPS), eta_min=1e-6
)
print(f"CosineAnnealingLR: T_max={EXPECTED_STEPS:,} steps")

def ckpt_path(): return Path(CFG["ckpt_dir"]) / "latest.pt"
def save_ckpt(model, opt, sched, epoch, step, rloss):
    tmp = ckpt_path().with_suffix(".tmp")
    torch.save({
        "model": model.state_dict(), "opt": opt.state_dict(),
        "scheduler": sched.state_dict(), "epoch": epoch,
        "step": step, "running_loss": rloss, "cfg": CFG
    }, tmp)
    tmp.replace(ckpt_path())

best_auc     = 0.0
best_epoch   = -1
start_epoch  = 0
global_step  = 0
running_loss = 0.0

if CFG["resume"] and ckpt_path().exists():
    obj = torch.load(ckpt_path(), map_location="cpu")
    # Validate bucket sizes match before loading
    saved_cfg = obj.get("cfg", {})
    if (saved_cfg.get("item_buckets") == CFG["item_buckets"] and
        saved_cfg.get("embed_dim")    == CFG["embed_dim"]):
        model.load_state_dict(obj["model"])
        opt.load_state_dict(obj["opt"])
        scheduler.load_state_dict(obj["scheduler"])
        start_epoch  = int(obj["epoch"])
        global_step  = int(obj["step"])
        running_loss = float(obj["running_loss"])
        print(f"Resumed: epoch={start_epoch}, step={global_step}")
    else:
        print("Checkpoint config mismatch — starting fresh.")
        print(f"  Saved: item_buckets={saved_cfg.get('item_buckets')}, "
              f"embed_dim={saved_cfg.get('embed_dim')}")
        print(f"  Current: item_buckets={CFG['item_buckets']}, "
              f"embed_dim={CFG['embed_dim']}")
else:
    print("Starting fresh.")

model.train()
history_log = []

for epoch in range(start_epoch, CFG["epochs"]):
    train_ds = SlidingWindowUMAGDataset(
        CFG, train_history, user_tier, ALL_ITEMS, epoch=epoch
    )
    loader = DataLoader(train_ds, batch_size=None, num_workers=0)
    pbar   = tqdm(loader, desc=f"Epoch {epoch}", mininterval=5.0)
    ep_loss = ep_n = 0
    t0 = time.time()

    for batch in pbar:
        u, i, t, s, sw, sh, shw, l, lw, tf, y = [x.to(device) for x in batch]

        opt.zero_grad(set_to_none=True)
        pred = model(u, i, t, s, sw, sh, shw, l, lw, tf)
        loss = loss_fn(pred, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        scheduler.step()

        lv = float(loss.item())
        running_loss = 0.98*running_loss + 0.02*lv if global_step > 0 else lv
        ep_loss += lv; ep_n += 1; global_step += 1

        if global_step % 500 == 0:
            pbar.set_postfix(
                ema=f"{running_loss:.4f}",
                lr=f"{scheduler.get_last_lr()[0]:.2e}",
            )
        if global_step % CFG["ckpt_every_steps"] == 0:
            save_ckpt(model, opt, scheduler, epoch, global_step, running_loss)

    val_auc, val_ndcg, val_hr, n_users = run_val(
        model, val_history, user_tier, ALL_ITEMS, CFG, device
    )
    dt = time.time() - t0
    entry = dict(
        epoch=epoch, loss=ep_loss/max(1,ep_n),
        val_auc=val_auc, val_ndcg10=val_ndcg, val_hr10=val_hr,
        batches=ep_n, time_min=dt/60
    )
    history_log.append(entry)
    print(f"Epoch {epoch} | loss={entry['loss']:.4f} | "
          f"val_AUC={val_auc:.4f} | NDCG@10={val_ndcg:.4f} | HR@10={val_hr:.4f} | "
          f"n_users={n_users} | batches={ep_n} | {dt/60:.1f}min")

    if val_auc > best_auc:
        best_auc   = val_auc
        best_epoch = epoch
        best_path  = Path(CFG["ckpt_dir"]) / "umag_books_v3_best.pt"
        torch.save({
            "model": model.state_dict(), "cfg": CFG,
            "epoch": epoch, "val_auc": val_auc,
            "val_ndcg10": val_ndcg, "val_hr10": val_hr
        }, best_path)
        print(f"  ★ New best AUC={best_auc:.4f} saved")

    save_ckpt(model, opt, scheduler, epoch+1, global_step, running_loss)

final_path = Path(CFG["ckpt_dir"]) / "umag_books_v3_final.pt"
torch.save({"model": model.state_dict(), "cfg": CFG}, final_path)
print(f"\nSaved: {final_path}")
print(f"Best epoch: {best_epoch} | Best AUC: {best_auc:.4f}")
print(f"DIEN Books benchmark: 0.8453")
print("\nTraining summary:")
for e in history_log:
    print(f"  Epoch {e['epoch']}: loss={e['loss']:.4f} | AUC={e['val_auc']:.4f} | "
          f"NDCG@10={e['val_ndcg10']:.4f} | HR@10={e['val_hr10']:.4f} | "
          f"{e['time_min']:.1f}min")

## Cell 9 — Evaluation + Gate Analysis (RQ2 / RQ3)


In [ ]:
import matplotlib.pyplot as plt

best_path = Path(CFG["ckpt_dir"]) / "umag_books_v3_best.pt"
if best_path.exists():
    obj = torch.load(best_path, map_location="cpu")
    model.load_state_dict(obj["model"])
    print(f"Loaded best checkpoint: epoch={obj['epoch']}, AUC={obj['val_auc']:.4f}")
model.eval()

rng_eval      = random.Random(99)
all_items_arr = ALL_ITEMS
n_items       = len(all_items_arr)
N_NEG_LOO     = 99
ub  = CFG["user_buckets"]
S   = CFG["session_window"]
M   = CFG["short_window"]
L   = CFG["long_window"]
ssl = M - S

all_preds, all_labels = [], []
all_gates, all_tiers  = [], []
eval_ranks = []

with torch.no_grad():
    for uid, hist in tqdm(val_history_full.items(), desc="Full eval"):
        if len(hist) < 2:
            continue

        u_hash    = stable_hash(uid, ub)
        t_idx     = user_tier.get(uid, 1)
        context   = hist[:-1]
        pos_entry = hist[-1]
        pos_item  = pos_entry[1]

        past_items = [h[1] for h in context]
        past_ws    = [h[2] for h in context]

        last_ts    = context[-1][0] if context else pos_entry[0]
        days_delta = max(0.0, (pos_entry[0] - last_ts) / SECS_PER_DAY)
        time_feat  = math.log1p(days_delta)

        seen_set  = set(h[1] for h in hist)
        neg_items = []
        attempts  = 0
        while len(neg_items) < N_NEG_LOO and attempts < N_NEG_LOO * 20:
            attempts += 1
            cand = all_items_arr[rng_eval.randrange(n_items)]
            if cand not in seen_set:
                neg_items.append(cand)
        if len(neg_items) < N_NEG_LOO:
            continue

        candidates = [pos_item] + neg_items
        n_cand     = len(candidates)

        sess_items  = past_items[-S:]
        sess_ws_    = past_ws[-S:]
        short_items = past_items[-(M):-S] if len(past_items) > S else []
        short_ws_   = past_ws[-(M):-S]    if len(past_ws)    > S else []
        long_items  = past_items[:-(M)]    if len(past_items) > M else []
        long_ws_    = past_ws[:-(M)]       if len(past_ws)    > M else []

        sess_pad  = pad_or_trim(sess_items,   S)
        short_pad = pad_or_trim(short_items, ssl)
        long_pad  = pad_or_trim(long_items,   L)
        sess_w_p  = pad_or_trim_w(sess_ws_,   S)
        short_w_p = pad_or_trim_w(short_ws_, ssl)
        long_w_p  = pad_or_trim_w(long_ws_,   L)

        if all(x == PAD for x in sess_pad):
            sess_pad = short_pad[:S] if len(short_pad) >= S \
                       else short_pad + [PAD]*(S-len(short_pad))
            sess_w_p = short_w_p[:S] if len(short_w_p) >= S \
                       else short_w_p + [0.0]*(S-len(short_w_p))

        u_t   = torch.tensor([u_hash]   * n_cand, dtype=torch.long).to(device)
        i_t   = torch.tensor(candidates,           dtype=torch.long).to(device)
        ti_t  = torch.tensor([t_idx]    * n_cand, dtype=torch.long).to(device)
        s_t   = torch.tensor([sess_pad]  * n_cand, dtype=torch.long).to(device)
        sw_t  = torch.tensor([sess_w_p]  * n_cand, dtype=torch.float32).to(device)
        sh_t  = torch.tensor([short_pad] * n_cand, dtype=torch.long).to(device)
        shw_t = torch.tensor([short_w_p] * n_cand, dtype=torch.float32).to(device)
        l_t   = torch.tensor([long_pad]  * n_cand, dtype=torch.long).to(device)
        lw_t  = torch.tensor([long_w_p]  * n_cand, dtype=torch.float32).to(device)
        tf_t  = torch.tensor([time_feat] * n_cand, dtype=torch.float32).to(device)

        pred, gates, _, _, _ = model(
            u_t, i_t, ti_t, s_t, sw_t, sh_t, shw_t, l_t, lw_t, tf_t,
            return_weights=True
        )
        scores_np = pred.cpu().numpy()
        gates_np  = gates.cpu().numpy()

        rank = int((scores_np[1:] >= scores_np[0]).sum())
        eval_ranks.append(rank)
        all_preds.extend(scores_np.tolist())
        all_labels.extend([1.0] + [0.0] * N_NEG_LOO)
        all_gates.append(gates_np[0])
        all_tiers.append(t_idx)

EP = np.array(all_preds)
EL = np.array(all_labels)
EG = np.array(all_gates)
ET = np.array(all_tiers)

auc          = roc_auc_score(EL, EP) if len(set(EL)) > 1 else float("nan")
ndcg10, hr10 = compute_ndcg_hr(eval_ranks, k=10)

print("\n" + "="*60)
print("UMAG v3 — AMAZON BOOKS (DIEN-aligned preprocessing)")
print("="*60)
print(f"  AUC-ROC    : {auc:.4f}  (DIEN benchmark: 0.8453)")
print(f"  NDCG@10    : {ndcg10:.4f}")
print(f"  HR@10      : {hr10:.4f}")
print(f"  Eval users : {len(eval_ranks):,}")

gate_means  = EG.mean(axis=0)
gate_stds   = EG.std(axis=0)
scale_names = [
    f"Session (last {CFG['session_window']})",
    f"Short   (next {CFG['short_window']-CFG['session_window']})",
    f"Long    (all older)",
]
labels_short = ["Session", "Short-term", "Long-term"]
dominant     = scale_names[int(np.argmax(gate_means))]

print("\n" + "="*60)
print("SCALE GATE WEIGHTS — RQ2")
print("="*60)
for name, m, s in zip(scale_names, gate_means, gate_stds):
    print(f"  {name:<32}: {m:.4f} ± {s:.4f}")
print(f"  Dominant scale: {dominant}")

electronics_gates = {"Session": 0.0942, "Short": 0.0899, "Long": 0.8159}
print("\n" + "="*60)
print("CROSS-DATASET GATE COMPARISON (Electronics v6 vs Books v3)")
print("="*60)
print(f"  {'Scale':<12} {'Electronics':>14} {'Books':>10} {'Δ':>8}")
print(f"  {'-'*46}")
for name, key, m in zip(labels_short, ["Session","Short","Long"], gate_means):
    elec = electronics_gates[key]
    print(f"  {name:<12} {elec:>14.4f} {m:>10.4f} {m-elec:>+8.4f}")

print("\n" + "="*60)
print("GATE WEIGHTS BY USER TIER — RQ3")
print("="*60)
tier_gate_summary = {}
for ti, tname in enumerate(TIER_NAMES):
    mask = (ET == ti)
    if mask.sum() == 0: continue
    tg = EG[mask].mean(axis=0)
    tier_gate_summary[tname] = tg
    print(f"  {tname:<8} (n={mask.sum():>6,}): "
          f"Session={tg[0]:.4f}  Short={tg[1]:.4f}  Long={tg[2]:.4f}")

# ── Plots ─────────────────────────────────────────────────────────────────────
colors = ["#378ADD", "#1D9E75", "#D85A30"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(labels_short, gate_means, color=colors, alpha=0.85)
axes[0].errorbar(labels_short, gate_means, yerr=gate_stds,
                 fmt="none", color="black", capsize=5)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Mean gate weight")
axes[0].set_title("Scale importance — Books v3 (RQ2)")

tier_colors = ["#9B59B6", "#3498DB", "#2ECC71", "#E74C3C"]
x = np.arange(3)
w = 0.2
for ti, tname in enumerate(TIER_NAMES):
    if tname not in tier_gate_summary: continue
    axes[1].bar(x + ti*w, tier_gate_summary[tname], w,
                label=tname, color=tier_colors[ti], alpha=0.85)
axes[1].set_xticks(x + w*1.5)
axes[1].set_xticklabels(labels_short)
axes[1].set_ylim(0, 1)
axes[1].set_title("Gate weights by user tier — Books v3 (RQ3)")
axes[1].legend(fontsize=8)

pos_scores = EP[EL == 1]
neg_scores = EP[EL == 0]
axes[2].hist(pos_scores, bins=40, alpha=0.6, label="Positive",
             color="#1D9E75", density=True)
axes[2].hist(neg_scores, bins=40, alpha=0.6, label="Negative",
             color="#D85A30", density=True)
axes[2].set_xlabel("Predicted score")
axes[2].set_title("Score distribution: pos vs neg")
axes[2].legend(fontsize=8)

plt.suptitle(
    f"UMAG v3 — Amazon Books (DIEN-aligned)\n"
    f"AUC={auc:.4f} | NDCG@10={ndcg10:.4f} | HR@10={hr10:.4f} "
    f"(DIEN benchmark: 0.8453)",
    fontsize=11, fontweight="bold"
)
plt.tight_layout()
plt.savefig("umag_books_v3_gate.png", dpi=150, bbox_inches="tight")
plt.show()

pd.DataFrame({
    "metric": ["auc_roc", "ndcg_at_10", "hr_at_10", "n_eval_users",
               "gate_session_mean", "gate_short_mean", "gate_long_mean",
               "dominant_scale", "dien_benchmark_auc",
               "preprocessing", "augmentation"],
    "value":  [round(auc,4), round(ndcg10,4), round(hr10,4), len(eval_ranks),
               round(float(gate_means[0]),4),
               round(float(gate_means[1]),4),
               round(float(gate_means[2]),4),
               dominant, 0.8453,
               "4star_plus_only", "sliding_window_all_prefixes"]
}).to_csv("umag_books_v3_metrics.csv", index=False)
print("\nSaved: umag_books_v3_metrics.csv, umag_books_v3_gate.png")